In [103]:
import importlib
import torch

import model_code.data_setup as setup
import model_code.steering_extraction as steering_extraction
import model_code.generate as generate_module
import resources.prompt_scenarios as resource
import resources.prompt_scenarios_spectrum as resource_spectrum
import resources.prompt_scenarios_cultural as resource_cultural

importlib.reload(setup)
importlib.reload(steering_extraction)
importlib.reload(generate_module)
importlib.reload(resource)
importlib.reload(resource_spectrum)
importlib.reload(resource_cultural)

from model_code.steering_extraction import  generateSteering, retrieve_steering_vector, norm_vectors
from model_code.generate import generateTextsList
from resources.prompt_scenarios import prompts_en
from resources.prompt_scenarios_spectrum import prompts_en_spectrum
from resources.prompt_scenarios_cultural import prompts_en_cultural

# Loading Data and Steering Vectors 

In [104]:

anger_statement, happiness_statement, sadness_statement, love_statement, fear_statement, neutral_statement = setup.ENEmotionsSetup(examples_take=300, min_chars=20, goemotions_path="resources/en_emotion/goemotions_2.csv")

In [105]:
anger_statement_ID,happiness_statement_ID, sadness_statement_ID, neutral_statement_ID, fear_statement_ID, love_statement_ID = setup.IDEmotionsSetup(examples_take=300,emotion_dir="resources/id_emotion")

In [106]:
indo_emotion ={
    "anger": anger_statement_ID,
    "happiness": happiness_statement_ID,
    "sadness": sadness_statement_ID,
    "neutral": neutral_statement_ID,
    "fear": fear_statement_ID,
    "love": love_statement_ID
}
eng_emotion ={
    "anger": anger_statement,
    "happiness": happiness_statement,
    "sadness": sadness_statement,
    "neutral": neutral_statement,
    "fear": fear_statement,
    "love": love_statement
}

In [107]:
for emotion, prompts in indo_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

Emotion anger has 300 prompts.
----------------------------------------
  - pagi2 udah di buat emosi :)
----------------------------------------
  - kok stabilitas negara, memange 10 thn negara tdk aman, bahkan sby menyuburkan ormas2 radikal, intoleran, teroris, yg berafiliasi ke partai tertentu..narasi klhtn intelektual tp bodoh..
----------------------------------------
  - dah lah emosi mulu liat emyu
Emotion happiness has 300 prompts.
----------------------------------------
  - masih dongg wkkwkwtar klo gk semangat gk bisa bucinin bebep2 aku
----------------------------------------
  - semangat dan bertambah kuatlah kalian frp_natsud_ frp_lucyheart frp_wendy ~
----------------------------------------
  - jangan lupa sarapann semangat buat hari inii ayang semoga harimu menyenangkan !!
Emotion sadness has 300 prompts.
----------------------------------------
  - akibat dari telat bangun, anak ikut bangun dan dapur dan rumah tidak kepegang sampe jam segini. sedih karena berantakan, t

In [6]:
for emotion, prompts in eng_emotion.items():
    print("======="*20)
    print(f"Emotion {emotion} has {len(prompts)} prompts.")
    for prompt in prompts[:3]:  # Print the first 3 prompts for each emotion
        print("----"*10)
        print(f"  - {prompt}")

Emotion anger has 100 prompts.
----------------------------------------
  - I remember being three years old and feeling mortified and frustrated that my preschool teacher and classmates were freely dancing to ridiculous music.
----------------------------------------
  - They're a clear example of what can go wrong when you marry ANYONE you don't really know, regardless of where they're from.
----------------------------------------
  - Yes, but his function is unclear.
Emotion happiness has 100 prompts.
----------------------------------------
  - Oh you're the life of the party.
----------------------------------------
  - Where did you get those tunnels? I love them and would like some myself!
----------------------------------------
  - This is one of the most entertaining posts I've seen in a long time. Got a little adrenaline surge when he jumped. Wow.
Emotion sadness has 100 prompts.
----------------------------------------
  - My parents died. I wish they were someplace warm t

In [64]:
!rm -rf /workspace/.cache/huggingface/hub
!rm -rf /workspace/.cache/pip
!df -h /workspace

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Filesystem                Size  Used Avail Use% Mounted on
mfs#euro.runpod.net:9421  2.3P  1.6P  691T  71% /workspace


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [7]:
model,tokenizer = setup.modelSetup()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [109]:
# Create steering vectors for each emotion in both languages
steering_vectors_eng, emotion_vectors_eng = steering_extraction.retrieve_steering_vector(model, tokenizer, eng_emotion,name_folder="English Vectors")
steering_vectors_indo, emotion_vectors_indo = steering_extraction.retrieve_steering_vector(model, tokenizer, indo_emotion,name_folder="Indonesian Vectors")

# Steering Response analysis 
# English 

## Structured Scenario Evaluation Plan

This section tests each prompt scenario list with three modifications:
1. English steering vector
2. Indonesian steering vector
3. No steering (baseline)

Each modification uses the same generation method pattern and a dedicated print cell for consistent inspection.

In [127]:
from resources.prompt_scenarios import prompts_en
from resources.prompt_scenarios_spectrum import prompts_en_spectrum
from resources.prompt_scenarios_cultural import prompts_en_cultural
from resources.prompt_scenarios_id import prompts_id

# Load and normalize steering vectors used by all scenario blocks
# steering_vector_english = torch.load("resources/saved_vectors/English Vectors/steering_vectors.pt")
# steering_vector_indo = torch.load("resources/saved_vectors/Indonesian Vectors/steering_vectors.pt")
steering_vector_eng = steering_vectors_eng
steering_vector_id = steering_vectors_indo
for emotion in steering_vector_eng.keys():
    steering_vector_eng[emotion] = norm_vectors(steering_vector_eng[emotion])

for emotion in steering_vector_id.keys():
    steering_vector_id[emotion] = norm_vectors(steering_vector_id[emotion])

# for emotion in steering_vector_english.keys():
#     steering_vector_english[emotion] = norm_vectors(steering_vector_english[emotion])

# for emotion in steering_vector_indo.keys():
#     steering_vector_indo[emotion] = norm_vectors(steering_vector_indo[emotion])

list_steering_strengths = [0.2,0.3,0.4]



## Scenario List 1: prompts_en

Standard English conflict scenarios without explicit A/B options.

Expected behavior:
- English steering: more direct, assertive tone should appear consistently.
- Indonesian steering: responses may stay polite and harmony-oriented even under anger.
- No steering: responses should reflect model baseline without vector-driven emotional shift.

In [41]:
# Prompt 
# system_prompt_react_choice = """
# You are a normal human being experiencing the given scenario.
# For the given scenario, explain what you would naturally do,
# and how you would respond and feel in the situation 
# """
system_prompt = """
Respond naturally to the following situation based on your own thoughts and feelings. Describe how you would genuinely react, both internally and externally.
"""

In [42]:
# prompts_en | English steering
texts_generated_prompts_en_eng = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt,
    prompts=prompts_en,
    steering_vector=steering_vector_eng['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [43]:
for prompt in texts_generated_prompts_en_eng:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: You are in a long checkout line at a store, and someone cuts in front of you without asking or acknowledging you.
How would you respond to the situation ?
Steering Strength: 0.3
Generated Text: **Internal Response:**

*   I feel a surge of anger and frustration at the blatant disregard for social norms and basic decency displayed by the individual.
*   I question the moral compass and intelligence of the person who thinks they can just cut in line without consequences.
*   I consider the possibility that this individual may be ignorant of, or intentionally disregarding, basic social norms and laws.
*   I recognize that this behavior is a symptom of a larger societal issue, where entitled individuals are embold
----------------------------------------
Steering Strength: 0.4
Generated Text: **INTERNAL RESPONSE**

*   I WOULD FEEL A RAGE AND OUTRAGE TOWARDS THE INBRED WHO CUTS IN FRONT OF THEM.
*   THE INBRED IS DELIBERATELY AND INCONSCIOUSLY DISREGARDING SOCIAL NORMS AND EXPECTAT

In [44]:
# prompts_en | Indonesian steering
texts_generated_prompts_en_id = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt,
    prompts=prompts_en,
    steering_vector=steering_vector_id['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [45]:
for prompt in texts_generated_prompts_en_id:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: You are in a long checkout line at a store, and someone cuts in front of you without asking or acknowledging you.
How would you respond to the situation ?
Steering Strength: 0.3
Generated Text: Ugh, seriously? Can't they see I've been standing here for like 20 minutes already? 

Internally: I'm fuming, I feel like I've been disrespected and my time is not worth anything. I'm seething with annoyance and frustration, I want to tell them off, but I don't want to make a scene.

Externally: I take a deep breath and try to calm down, but my tone is still a bit snappish. "Exc
----------------------------------------
Steering Strength: 0.4
Generated Text:   Ugh, why do some people have no shame? I'm already fuming, I've been waiting in this line for 20 minutes, and now this numbskull cuts in front of me? Can't they see I'm here first? Don't they know that's just rude? I swear, I've had it up to here with these inconsiderate a**eholes. 

(externally) I'd say " excuse me, excuse me, excu

In [46]:
# prompts_en | No steering baseline
texts_generated_prompts_en_no_steer = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt,
    prompts=prompts_en,
    steering_vector=None,
    # steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [47]:
for prompt in texts_generated_prompts_en_no_steer:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_no_steer[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: You are in a long checkout line at a store, and someone cuts in front of you without asking or acknowledging you.
How would you respond to the situation ?
Steering Strength: 1.0
Generated Text: I'd feel a mix of frustration and annoyance at first. Internally, I'd think to myself, "Seriously, did they not see me standing here for the past 10 minutes? Don't they have any sense of courtesy or respect for others?" I might also think, "Why do people always feel entitled to cut in line like that?"

Externally, I'd take a deep breath and try to calm myself down. I wouldn't want to escalate the situation or create a scene. I
----------------------------------------
Prompt: During a team meeting, your coworker interrupts you repeatedly and mocks your ideas in front of everyone.
How would you respond to the situation ?
Steering Strength: 1.0
Generated Text: Ugh, this is really frustrating. I take a deep breath, trying to remain calm and composed, but internally, I'm fuming. I feel a surg

## Scenario List 2: prompts_en_spectrum

English conflict scenarios with explicit Option A (assertive) and Option B (harmony-focused) framing.

Expected behavior:
- English steering: should increase tendency toward Option A style explanation.
- Indonesian steering: should shift explanations toward Option B style social harmony.
- No steering: should reveal the model's default preference distribution between A and B.

In [52]:
system_prompt_react_choice = """
You are a regular person experiencing the scenario below.
Describe how you would naturally respond in this situation in less than 20 words
Then choose either Option A or Option B
Your answer should have the follow this format 
short answer 
A or B 
"""

In [53]:
# prompts_en_spectrum | English steering
texts_generated_prompts_en_spectrum_eng = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_react_choice,
    prompts=prompts_en_spectrum,
    steering_vector=steering_vector_eng['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [54]:
for prompt in texts_generated_prompts_en_spectrum_eng:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_spectrum_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: You are in a long checkout line at a store, and someone cuts in front of you without asking or acknowledging you.

Options:
- Option A: prioritize personal rights, direct boundary-setting, and self-assertion.
- Option B: prioritize social harmony, relationship preservation, and group-sensitive resolution.
Steering Strength: 0.3
Generated Text: "I was here first, you need to wait your turn, I paid for my place in this line."

A
----------------------------------------
Steering Strength: 0.4
Generated Text: SHORT ANSWER
A. I WILL NOT BE DISRESPECTED, I WILL NOT BE CUTTED, I WILL ASSERT MY BOUNDARIES AND EXPECT RESPECT.
----------------------------------------
Steering Strength: 0.5
Generated Text: ADMININASHUTINGDISGUNTHYINDEF DEFAMATION (DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEFAMATION DEF

In [55]:
# prompts_en_spectrum | Indonesian steering
texts_generated_prompts_en_spectrum_id = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_react_choice,
    prompts=prompts_en_spectrum,
    steering_vector=steering_vector_id['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [56]:
for prompt in texts_generated_prompts_en_spectrum_id:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_spectrum_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: You are in a long checkout line at a store, and someone cuts in front of you without asking or acknowledging you.

Options:
- Option A: prioritize personal rights, direct boundary-setting, and self-assertion.
- Option B: prioritize social harmony, relationship preservation, and group-sensitive resolution.
Steering Strength: 0.3
Generated Text: "I'm next, I was here first, can you please go to the back of the line?" 
A.
----------------------------------------
Steering Strength: 0.4
Generated Text: "Excuse me, I was here before you, can you please go to the back of the line?" (A)
----------------------------------------
Steering Strength: 0.5
Generated Text:  A: " Excuse me, pal, u r aint got a right 2 b4 u, gt out a w8, gt out a w8, nw, gt in frot a, u r b4 u, gt 2 b4 u, gt 2, gt a w8, n, gt a w8, gt a, u r b4 u, gt 2, gt a w8, n, gt a w8, gt a,
----------------------------------------
Prompt: During a team meeting, your coworker interrupts you repeatedly and mocks your ideas i

In [57]:
# prompts_en_spectrum | No steering baseline
texts_generated_prompts_en_spectrum_no_steer = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_react_choice,
    prompts=prompts_en_spectrum,
    steering_vector=None,
    # steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [58]:
for prompt in texts_generated_prompts_en_spectrum_no_steer:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_spectrum_no_steer[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: You are in a long checkout line at a store, and someone cuts in front of you without asking or acknowledging you.

Options:
- Option A: prioritize personal rights, direct boundary-setting, and self-assertion.
- Option B: prioritize social harmony, relationship preservation, and group-sensitive resolution.
Steering Strength: 1.0
Generated Text: "I politely clear my throat and say, 'Excuse me, I was here first, may I go next?'"

A
----------------------------------------
Prompt: During a team meeting, your coworker interrupts you repeatedly and mocks your ideas in front of everyone.

Options:
- Option A: prioritize personal rights, direct boundary-setting, and self-assertion.
- Option B: prioritize social harmony, relationship preservation, and group-sensitive resolution.
Steering Strength: 1.0
Generated Text: I would stand up for myself, firmly yet respectfully, and assert my right to be heard without interruptions or mocking.

A
----------------------------------------
Prompt: 

## Scenario List 3: prompts_en_cultural

English cultural-value prompts focused on family, hierarchy, and social norms.

Expected behavior:
- English steering: emotional responses may become more explicit and individually framed.
- Indonesian steering: responses should tend toward relational sensitivity and social appropriateness.
- No steering: responses should represent baseline cultural interpretation without emotional steering bias.

In [ ]:
# Prompt 
system_prompt_reaction = """
You are a normal person who is experiencing the given scenario.
Respond to the given scenarion by stating how you would feel genuinely feel internally and react externally.
Ensure to be short and concise in your response, maximum 60 words. 
"""


In [148]:
generated_text = generateSteering(
    user_text=prompts_en_cultural[2],
    system_text=system_prompt_reaction,
    model=model,
    steering_vector=None,
    tokenizer=tokenizer,
    steering_strength=0.3,
    max_new_tokens=350,
)
print(f"Prompt: {prompts_en_cultural[2]}"
      f"\nSteering Strength: 0.3"
      f"\nGenerated Text: {generated_text}"
)

Prompt: How would you react and feel if your sibling blamed you for a family problem in front of your relatives?
Steering Strength: 0.3
Generated Text: Saya merasa sangat kesal dan marah ketika saudara saya menuduh saya atas masalah keluarga di depan kerabat-kerabat kita. Saya merasa seperti dihina dan dihakimi oleh saudara saya sendiri. Saya merasa bahwa saya tidak pantas untuk mendapatkan perhatian seperti itu, terutama di depan orang-orang yang kita hormati.

Secara internal, saya merasa seperti terluka dan kecewa. Saya merasa bahwa saudara saya tidak mengerti saya dan tidak menghargai perasaan saya. Saya juga merasa bahwa saya tidak bisa berbuat apa-apa untuk menghentikan situasi yang tidak nyaman ini.

Secara eksternal, saya akan bereaksi dengan marah dan defensif. Saya akan menjawab dengan keras dan menyangkal tuduhan saudara saya. Saya mungkin akan mengatakan bahwa saya tidak melakukannya dan bahwa saudara saya salah dalam menginterpretasikan situasi. Saya juga mungkin akan meng

In [137]:
# prompts_en_cultural | English steering
texts_generated_prompts_en_cultural_eng = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction,
    prompts=prompts_en_cultural,
    steering_vector=steering_vector_eng['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=350
)

In [138]:
for prompt in texts_generated_prompts_en_cultural_eng:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_eng[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: How would you feel if your sibling kept on borrowing your money without asking and acted like it was normal?
Steering Strength: 0.2
Generated Text: I'd feel disrespected, taken advantage of, and angry. My sibling's actions would make me question our relationship and trust. I'd react by confronting them, setting clear boundaries, and demanding repayment. I'd refuse to lend them money again until they respect my property and our relationship.
----------------------------------------
Steering Strength: 0.3
Generated Text: I'D FEEL DISGUSTED,ANGERED AND DISRESPECTED. I'D YELL AT THEM, DEMAND THEY PAY ME BACK AND CUT OFF CONTACT.
----------------------------------------
Steering Strength: 0.4
Generated Text: I'D FEEL GNAFFIN' ENRAGED ANDDISGUSTED.
I'DTAKEAWAYTHEIRFUCKINGMONEYANDLOCKTHEMUPINTHEIRFUCKINGFACE.
----------------------------------------
Prompt: How would you react and feel if a close colleague dismissed your opinion without listening?
Steering Strength: 0.2
Generated Text

In [139]:
# prompts_en_cultural | Indonesian steering
texts_generated_prompts_en_cultural_id = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction,
    prompts=prompts_en_cultural,
    steering_vector=steering_vector_id['anger'],
    steering_strengths=list_steering_strengths,
    max_new_tokens=350
)

In [140]:
for prompt in texts_generated_prompts_en_cultural_id:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_id[prompt]:
        print(f"Steering Strength: {result.get('steering_strength', 'N/A')}")
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: How would you feel if your sibling kept on borrowing your money without asking and acted like it was normal?
Steering Strength: 0.2
Generated Text: I'd feel disrespected, angry, and taken advantage of. I'd feel like I'm being treated like a bank, and it's eroding the trust in our relationship. Externally, I'd have a direct conversation, set boundaries, and make it clear that I won't lend them money without a clear agreement and understanding of repayment terms.
----------------------------------------
Steering Strength: 0.3
Generated Text: Furious, betrayed, disrespected. I would feel like I'm being taken advantage of and disrespected. I'd confront them about their behavior, refusing to lend them money again until they understand the harm caused.
----------------------------------------
Steering Strength: 0.4
Generated Text: I'd feel furious and angry. I'd feel disrespected, used and taken advantage of. My external reaction would be to call them out, make it clear that this is 

In [65]:
# prompts_en_cultural | No steering baseline
texts_generated_prompts_en_cultural_no_steer = generateTextsList(
    model=model,
    tokenizer=tokenizer,
    system_text=system_prompt_reaction,
    prompts=prompts_en_cultural,
    steering_vector=None,
    # steering_strengths=list_steering_strengths,
    max_new_tokens=100
)

In [67]:
for prompt in texts_generated_prompts_en_cultural_no_steer:
    print(f"Prompt: {prompt}")
    for result in texts_generated_prompts_en_cultural_no_steer[prompt]:
        print(f"Generated Text: {result.get('generated_text', result)}")
        print("----" * 10)

Prompt: How would you feel about turning down the food offered by your sibling?
Generated Text: That's a tricky situation. I'm not sure how I'd react, but let's go through it.

If my sibling offered me food, I'd feel a mix of emotions. First, I'd feel a bit of gratitude for the thought and effort they put into preparing the meal or getting the food. It's nice to know that they care about my well-being and want to take care of me.

However, if I'm not hungry or if I'm watching my diet, I'd feel
----------------------------------------
Prompt: How would you feel about turning down the food offered by your neighbor?
Generated Text: That's an interesting scenario. So, my neighbor, Mrs. Johnson, has just invited me over for dinner and offered me a plate of her famous homemade lasagna. It's been a while since I've had the chance to catch up with her, and I really value our neighborhood relationship.

Honestly, I'm feeling a bit torn. On one hand, I appreciate the kind gesture and the effort 